# 19_documentation_pack.ipynb — Pack de documentación técnica

Este notebook **no entrena modelos**. Genera documentación lista para GitHub, memoria y presentación a partir de los resultados ya consolidados.

Salidas principales:

```text
README.md
.gitignore
docs/*.md
reports/model_summaries/
reports/tables/
reports/figures/
```

Preparado para ejecutar en **Mac** desde `deep-wave-canarias/notebooks/`.

## Celda 0 — Entorno

In [1]:
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Entorno local detectado. No se monta Google Drive.")

Entorno local detectado. No se monta Google Drive.


## Celda 1 — Imports, rutas y configuración

In [2]:
from pathlib import Path
import sys, json, shutil, platform
from datetime import datetime
import pandas as pd
import numpy as np

RUNNING_IN_COLAB = "google.colab" in sys.modules
BASE_DIR = Path("/content/drive/MyDrive/AI Projects/DeepWave Canarias") if RUNNING_IN_COLAB else Path.cwd().parent.resolve()

GOLD_DIR = BASE_DIR / "gold"
FINAL_RESULTS_DIR = GOLD_DIR / "model_results" / "final_report_multitarget"

DOCS_DIR = BASE_DIR / "docs"
REPORTS_DIR = BASE_DIR / "reports"
REPORTS_TABLES_DIR = REPORTS_DIR / "tables"
REPORTS_FIGURES_DIR = REPORTS_DIR / "figures"
REPORTS_SUMMARIES_DIR = REPORTS_DIR / "model_summaries"

README_PATH = BASE_DIR / "README.md"
GITIGNORE_PATH = BASE_DIR / ".gitignore"

OVERWRITE_DOCS = True
BACKUP_EXISTING = True

for d in [DOCS_DIR, REPORTS_DIR, REPORTS_TABLES_DIR, REPORTS_FIGURES_DIR, REPORTS_SUMMARIES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Sistema:", platform.platform())
print("BASE_DIR:", BASE_DIR)
print("FINAL_RESULTS_DIR:", FINAL_RESULTS_DIR, "existe:", FINAL_RESULTS_DIR.exists())
print("DOCS_DIR:", DOCS_DIR)

Sistema: macOS-26.3.1-arm64-arm-64bit
BASE_DIR: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias
FINAL_RESULTS_DIR: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/gold/model_results/final_report_multitarget existe: True
DOCS_DIR: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs


## Celda 2 — Funciones auxiliares

In [3]:
generated_files = []

def read_text_safe(path):
    path = Path(path)
    if not path.exists():
        return ""
    try:
        return path.read_text(encoding="utf-8")
    except Exception:
        return ""


def read_json_safe(path):
    path = Path(path)
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}


def read_csv_safe(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except Exception as e:
        print("AVISO: no se pudo leer", path, e)
        return pd.DataFrame()


def backup_file(path):
    path = Path(path)
    if path.exists() and BACKUP_EXISTING:
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup = path.with_suffix(path.suffix + f".bak_{stamp}")
        shutil.copy2(path, backup)
        return backup
    return None


def write_doc(path, content, overwrite=OVERWRITE_DOCS):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and not overwrite:
        print("SKIP existe:", path)
        return False
    if path.exists() and overwrite:
        backup_file(path)
    path.write_text(content.strip() + "\n", encoding="utf-8")
    generated_files.append(str(path))
    print("Escrito:", path)
    return True


def copy_if_exists(src, dst_dir, overwrite=True):
    src, dst_dir = Path(src), Path(dst_dir)
    if not src.exists():
        return None
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / src.name
    if dst.exists() and overwrite:
        backup_file(dst)
    if not dst.exists() or overwrite:
        shutil.copy2(src, dst)
        generated_files.append(str(dst))
        print("Copiado:", src, "->", dst)
    return dst


def md_table(rows, columns):
    if not rows:
        return "_No disponible._"
    def fmt(x):
        if x is None:
            return ""
        try:
            if pd.isna(x):
                return ""
        except Exception:
            pass
        if isinstance(x, float):
            return f"{x:.4f}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    for r in rows:
        lines.append("| " + " | ".join(fmt(r.get(c, "")) for c in columns) + " |")
    return "\n".join(lines)


def short_metric(df, filters, value_col, default="NA"):
    if df is None or df.empty or value_col not in df.columns:
        return default
    work = df.copy()
    for col, val in filters.items():
        if col not in work.columns:
            return default
        work = work[work[col] == val]
    if work.empty:
        return default
    val = work.iloc[0][value_col]
    try:
        if pd.isna(val): return default
        if isinstance(val, (float, np.floating)): return f"{float(val):.4f}"
        return str(val)
    except Exception:
        return default

## Celda 3 — Cargar resultados finales

In [4]:
final_architecture = read_json_safe(FINAL_RESULTS_DIR / "final_architecture_decision.json")
final_report_md = read_text_safe(FINAL_RESULTS_DIR / "final_modeling_report.md")
final_short_summary_md = read_text_safe(FINAL_RESULTS_DIR / "final_modeling_short_summary.md")

final_executive = read_csv_safe(FINAL_RESULTS_DIR / "tables" / "final_executive_results_table.csv")
final_modeling_status = read_csv_safe(FINAL_RESULTS_DIR / "tables" / "final_modeling_status.csv")
final_physical_scalar = read_csv_safe(FINAL_RESULTS_DIR / "tables" / "final_physical_scalar_improvement.csv")
final_physical_direction = read_csv_safe(FINAL_RESULTS_DIR / "tables" / "final_physical_direction_improvement.csv")
final_risk_recommendations = read_csv_safe(FINAL_RESULTS_DIR / "tables" / "final_risk_recommendations.csv")
final_surf_recommendations = read_csv_safe(FINAL_RESULTS_DIR / "tables" / "final_surf_recommendations.csv")

hs_24_mae = short_metric(final_physical_scalar, {"target_name": "hs", "horizon_hours": 24}, "mae_model")
hs_48_mae = short_metric(final_physical_scalar, {"target_name": "hs", "horizon_hours": 48}, "mae_model")

print("final_executive:", final_executive.shape)
print("final_modeling_status:", final_modeling_status.shape)
print("final_physical_scalar:", final_physical_scalar.shape)
print("final_risk_recommendations:", final_risk_recommendations.shape)
print("final_surf_recommendations:", final_surf_recommendations.shape)
print("MAE hs +24h:", hs_24_mae)
print("MAE hs +48h:", hs_48_mae)

final_executive: (65, 9)
final_modeling_status: (3, 6)
final_physical_scalar: (60, 11)
final_risk_recommendations: (15, 9)
final_surf_recommendations: (5, 8)
MAE hs +24h: 0.3159
MAE hs +48h: 0.4432


## Celda 4 — Crear .gitignore

In [5]:
gitignore_content = r"""
# Python
__pycache__/
*.py[cod]
.venv/
venv/
env/
.ipynb_checkpoints/

# Mac
.DS_Store

# Secretos / entorno
.env
.env.*
secrets/
credentials/
token*.json
*.key
*.pem

# Datos y modelos pesados
data/
bronze/
silver/
gold/
models/

# Archivos pesados
*.parquet
*.nc
*.grib
*.grib2
*.h5
*.hdf5
*.pkl
*.joblib
*.zip
*.tar
*.gz
*.7z

# CSV grandes por defecto
*.csv
!reports/**/*.csv
!docs/**/*.csv

# Logs y temporales
logs/
*.log
tmp/
temp/
cache/
"""
write_doc(GITIGNORE_PATH, gitignore_content, overwrite=True)

Escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/.gitignore


True

## Celda 5 — Crear README.md

In [6]:
readme_content = f"""
# DeepWave Canarias

Sistema predictivo de oleaje, viento, marea y riesgo marítimo para zonas costeras de Canarias mediante Inteligencia Artificial y Big Data.

## Objetivo

DeepWave Canarias anticipa el estado marítimo futuro en zonas costeras de Canarias y transforma predicciones físicas en información operativa para baño, surf, navegación ligera y seguridad costera.

Horizontes de predicción:

```text
+3h, +6h, +12h, +24h y +48h
```

## Qué predice

Variables físicas principales:

- altura significativa de ola (`hs`)
- periodo medio (`tm02`)
- swell height / swell period
- dirección de ola
- velocidad y dirección del viento
- nivel del mar y rango mareal

Módulos derivados:

- riesgo general marítimo
- riesgo para playa/bañistas
- riesgo para navegación ligera
- surf score de 0 a 10 y categoría de calidad

## Arquitectura de datos

```text
Bronze → Silver → Gold → Gold multitarget
```

## Arquitectura final de modelado

```text
Predicción física con LightGBM
→ riesgo interpretable
→ surf score interpretable
```

## Resultados destacados

Ejemplos para `hs`:

- MAE +24h: `{hs_24_mae}` m
- MAE +48h: `{hs_48_mae}` m

## Estructura del repositorio

```text
deep-wave-canarias/
├── notebooks/
├── docs/
├── reports/
│   ├── figures/
│   ├── tables/
│   └── model_summaries/
├── requirements.txt
├── requirements-mac-m2pro.txt
├── README.md
└── .gitignore
```

## Importante

No se suben a GitHub datos pesados ni modelos:

```text
data/
silver/
gold/
models/
```

## Limitaciones

- Las etiquetas de riesgo y surf son derivadas mediante reglas físicas.
- No se dispone de etiquetas oficiales de incidentes, banderas o valoraciones reales de surfistas.
- El horizonte +48h tiene mayor incertidumbre.
- El sistema es complementario y no sustituye avisos oficiales.
"""
write_doc(README_PATH, readme_content, overwrite=True)

Escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/README.md


True

## Celda 6 — Crear documentos principales

In [7]:
docs = {}

docs["01_contexto_y_objetivos.md"] = """
# 01 — Contexto y objetivos

## Contexto

Las Islas Canarias presentan alta variabilidad marítima por oleaje atlántico, vientos alisios, batimetría compleja y orientación local de la costa.

## Problema

La información marítima general no siempre describe bien el riesgo local por playa o zona costera.

## Objetivo general

Desarrollar un sistema predictivo capaz de anticipar condiciones marítimas futuras y transformarlas en indicadores de riesgo y utilidad.

## Objetivos específicos

1. Recopilar datos oceanográficos y meteorológicos.
2. Construir un pipeline Bronze/Silver/Gold.
3. Entrenar modelos físicos.
4. Evaluar riesgo marítimo.
5. Evaluar surf score.
6. Consolidar resultados para memoria y presentación.
"""

docs["02_fuentes_de_datos.md"] = """
# 02 — Fuentes de datos

Fuentes principales:

- Puertos del Estado: oleaje, viento marítimo, nivel del mar y redes SIMAR/REDMAR/REDEXT/REDCOS.
- AEMET OpenData: variables meteorológicas.
- ERA5 / GFS / Copernicus: reanálisis y predicción atmosférica/oceanográfica.
- GEBCO / EMODnet: batimetría.

Las fuentes se integran por zona y timestamp tras limpieza en Silver.
"""

docs["03_pipeline_bronze_silver_gold.md"] = """
# 03 — Pipeline Bronze, Silver y Gold

## Bronze

Datos originales sin transformación destructiva.

## Silver

Datos limpios por fuente: fechas, unidades, sentinelas, rangos físicos, duplicados y asignación espacial.

## Gold

Dataset final de entrenamiento con features, lags, rolling windows, targets y splits temporales.

## Gold multitarget

Dataset ampliado para predicción física, riesgo y surf score.
"""

docs["04_datasets_y_variables.md"] = """
# 04 — Datasets y variables

## Gold inicial

`gold/training_dataset/`

## Gold multitarget

`gold/multitarget_training_dataset/`

## Variables principales

- hs
- tm02
- swell_height
- swell_period
- wave_direction
- wind_speed
- wind_direction
- sea_level
- daily_tidal_range

## Targets

Horizontes:

```text
+3h, +6h, +12h, +24h, +48h
```

Ejemplos:

```text
target_hs_24h
target_risk_beach_12h
target_surf_score_48h
```
"""

docs["05_modelos_entrenados.md"] = """
# 05 — Modelos entrenados

## Modelo físico principal

LightGBMRegressor por variable y horizonte.

## Variables direccionales

Se entrenan como:

```text
sin(dirección)
cos(dirección)
```

## Riesgo

Se evaluaron clasificadores directos y riesgo derivado desde predicciones físicas.

## Surf score

Se compararon condiciones actuales, score derivado físico y modelo directo LightGBM.

## Decisión final

La arquitectura final prioriza:

```text
predicción física → reglas interpretables
```
"""

# Resultados con tablas dinámicas.
physical_rows = []
if not final_physical_scalar.empty:
    for target in ["hs", "tm02", "swell_height", "wind_speed", "sea_level"]:
        sub = final_physical_scalar[final_physical_scalar.get("target_name", "") == target]
        for _, r in sub.head(5).iterrows():
            physical_rows.append({"target": target, "horizon": r.get("horizon_hours", ""), "mae": r.get("mae_model", ""), "improvement_pct": r.get("mae_improvement_pct", "")})

risk_rows = []
if not final_risk_recommendations.empty:
    for _, r in final_risk_recommendations.iterrows():
        risk_rows.append({"module": r.get("risk_module", ""), "horizon": r.get("horizon_hours", ""), "method": r.get("recommended_model", ""), "score": r.get("recommended_score", "")})

surf_rows = []
if not final_surf_recommendations.empty:
    for _, r in final_surf_recommendations.iterrows():
        surf_rows.append({"horizon": r.get("horizon_hours", ""), "recommended": r.get("recommended_model", ""), "mae": r.get("recommended_mae", ""), "macro_f1_quality": r.get("recommended_quality_macro_f1", "")})

docs["06_resultados_modelado.md"] = f"""
# 06 — Resultados de modelado

## Resultado destacado de hs

- MAE +24h: {hs_24_mae} m
- MAE +48h: {hs_48_mae} m

## Resultados físicos resumidos

{md_table(physical_rows[:30], ["target", "horizon", "mae", "improvement_pct"])}

## Riesgo resumido

{md_table(risk_rows[:30], ["module", "horizon", "method", "score"])}

## Surf score resumido

{md_table(surf_rows, ["horizon", "recommended", "mae", "macro_f1_quality"])}
"""

docs["07_arquitectura_final.md"] = """
# 07 — Arquitectura final

La arquitectura final queda:

```text
Gold multitarget
    ↓
LightGBM físico
    ↓
Predicciones de hs, periodo, viento, dirección y marea
    ↓
Reglas físicas interpretables
    ↓
Riesgo general / playa / navegación / surf score
```

Esta arquitectura es más explicable que entrenar modelos directos sobre etiquetas derivadas por reglas.
"""

docs["08_limitaciones_y_trabajo_futuro.md"] = """
# 08 — Limitaciones y trabajo futuro

## Limitaciones

- Riesgo y surf score se derivan mediante reglas físicas.
- No hay etiquetas oficiales de incidentes, banderas o calidad real de surf.
- +48h tiene mayor incertidumbre.
- El uso de zona_id limita la extrapolación directa a zonas nuevas.
- No sustituye avisos oficiales.

## Trabajo futuro

- Incorporar etiquetas reales de playa y navegación.
- Añadir más zonas.
- Añadir intervalos de confianza.
- Desplegar API y dashboard.
- Validar con usuarios o expertos.
"""

docs["09_reproducibilidad.md"] = """
# 09 — Reproducibilidad

## Entorno recomendado

```text
Mac M2 Pro
Python 3.11
VS Code / Jupyter
```

## Instalación

```bash
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r requirements-mac-m2pro.txt
```

Si LightGBM falla en Mac:

```bash
brew install libomp
python -m pip install lightgbm
```

## Orden final recomendado

```text
14_gold_multitarget_dataset.ipynb
15_model_training_multitarget_physical.ipynb
16_model_training_risk_modules.ipynb
17_model_surf_score.ipynb
18_model_final_report_multitarget.ipynb
19_documentation_pack.ipynb
```
"""

for name, content in docs.items():
    write_doc(DOCS_DIR / name, content)

Escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/01_contexto_y_objetivos.md
Escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/02_fuentes_de_datos.md
Escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/03_pipeline_bronze_silver_gold.md
Escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/04_datasets_y_variables.md
Escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/05_modelos_entrenados.md
Escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/06_resultados_modelado.md
Escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/07_arquitectura_final.md
Escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/08_limitaciones_y_trabajo_futuro.md
Escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/09_reproduci

## Celda 7 — Índice de notebooks, memoria, GitHub y bibliografía

In [8]:
notebooks_rows = [
    {"notebook": "10_silver_quality_audit.ipynb", "objetivo": "Auditoría Silver", "estado": "OK"},
    {"notebook": "11_gold_training_dataset.ipynb", "objetivo": "Gold inicial", "estado": "OK"},
    {"notebook": "12_model_training_lightgbm.ipynb", "objetivo": "Modelo hs", "estado": "OK"},
    {"notebook": "12B_model_training_xgboost_risk.ipynb", "objetivo": "Riesgo inicial XGBoost", "estado": "OK"},
    {"notebook": "13_model_training_tcn.ipynb", "objetivo": "TCN experimental", "estado": "Experimental"},
    {"notebook": "14_gold_multitarget_dataset.ipynb", "objetivo": "Gold multitarget", "estado": "OK"},
    {"notebook": "15_model_training_multitarget_physical.ipynb", "objetivo": "Modelos físicos", "estado": "OK"},
    {"notebook": "16_model_training_risk_modules.ipynb", "objetivo": "Riesgo", "estado": "OK"},
    {"notebook": "17_model_surf_score.ipynb", "objetivo": "Surf score", "estado": "OK"},
    {"notebook": "18_model_final_report_multitarget.ipynb", "objetivo": "Informe final modelado", "estado": "OK"},
    {"notebook": "19_documentation_pack.ipynb", "objetivo": "Documentación", "estado": "OK"},
]

notebooks_doc = f"""
# Índice de notebooks

{md_table(notebooks_rows, ["notebook", "objetivo", "estado"])}
"""

memory_outline = """
# Estructura recomendada de la memoria

1. Introducción
2. Objetivos
3. Contexto y justificación
4. Fuentes de datos
5. Pipeline Bronze/Silver/Gold
6. Calidad del dataset
7. Modelos predictivos
8. Evaluación
9. Resultados
10. Rendimiento técnico
11. Limitaciones
12. Conclusiones
13. Bibliografía
"""

github_checklist = """
# Checklist GitHub

- [ ] Revisar `.gitignore`.
- [ ] No subir `data/`, `silver/`, `gold/`, `models/`.
- [ ] No subir `.pkl`, `.parquet`, `.nc`.
- [ ] Subir `README.md`, `docs/`, `reports/`, `notebooks/`.
- [ ] Confirmar que no hay secretos ni tokens.

Comandos:

```bash
git status
git add README.md .gitignore docs/ reports/ notebooks/ requirements.txt requirements-mac-m2pro.txt
git commit -m "Add documentation pack"
git push
```
"""

bibliografia = """
# Bibliografía y webgrafía base

## Fuentes

- Puertos del Estado
- AEMET OpenData
- Copernicus Marine Service
- ERA5 / Copernicus Climate Data Store
- NOAA GFS
- GEBCO / EMODnet Bathymetry

## Librerías

- pandas
- NumPy
- PyArrow
- scikit-learn
- LightGBM
- XGBoost
- Matplotlib
- Jupyter

Completar en la memoria con URLs exactas y fecha de consulta.
"""

write_doc(DOCS_DIR / "notebooks_index.md", notebooks_doc)
write_doc(DOCS_DIR / "memoria_outline.md", memory_outline)
write_doc(DOCS_DIR / "github_checklist.md", github_checklist)
write_doc(DOCS_DIR / "bibliografia_base.md", bibliografia)

Escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/notebooks_index.md
Escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/memoria_outline.md
Escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/github_checklist.md
Escrito: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/bibliografia_base.md


True

## Celda 8 — Copiar informes, tablas y figuras finales ligeras

In [9]:
for src in [
    FINAL_RESULTS_DIR / "final_modeling_report.md",
    FINAL_RESULTS_DIR / "final_modeling_short_summary.md",
    FINAL_RESULTS_DIR / "final_architecture_decision.json",
]:
    copy_if_exists(src, REPORTS_SUMMARIES_DIR, overwrite=True)

final_tables_dir = FINAL_RESULTS_DIR / "tables"
if final_tables_dir.exists():
    for src in sorted(final_tables_dir.glob("*.csv")):
        copy_if_exists(src, REPORTS_TABLES_DIR, overwrite=True)

final_figures_dir = FINAL_RESULTS_DIR / "figures"
if final_figures_dir.exists():
    for src in sorted(final_figures_dir.glob("*.png")):
        copy_if_exists(src, REPORTS_FIGURES_DIR, overwrite=True)

print("Copia completada.")

Copiado: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/gold/model_results/final_report_multitarget/final_modeling_report.md -> /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/reports/model_summaries/final_modeling_report.md
Copiado: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/gold/model_results/final_report_multitarget/final_modeling_short_summary.md -> /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/reports/model_summaries/final_modeling_short_summary.md
Copiado: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/gold/model_results/final_report_multitarget/final_architecture_decision.json -> /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/reports/model_summaries/final_architecture_decision.json
Copiado: /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/gold/model_results/final_report_multitarget/tables/copied_model_reports.cs

## Celda 9 — Manifest y validación final

In [10]:
manifest_rows = []
seen = set()

for path_str in generated_files:
    p = Path(path_str)
    if p.exists():
        rel = str(p.relative_to(BASE_DIR)) if str(p).startswith(str(BASE_DIR)) else str(p)
        seen.add(rel)
        manifest_rows.append({
            "path": rel,
            "size_kb": round(p.stat().st_size / 1024, 2),
            "modified": datetime.fromtimestamp(p.stat().st_mtime).isoformat(timespec="seconds"),
        })

for p in sorted(DOCS_DIR.glob("*.md")):
    rel = str(p.relative_to(BASE_DIR))
    if rel not in seen:
        manifest_rows.append({
            "path": rel,
            "size_kb": round(p.stat().st_size / 1024, 2),
            "modified": datetime.fromtimestamp(p.stat().st_mtime).isoformat(timespec="seconds"),
        })

manifest_df = pd.DataFrame(manifest_rows).sort_values("path").reset_index(drop=True)
manifest_path = DOCS_DIR / "documentation_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

display(manifest_df)

required_files = [
    README_PATH,
    GITIGNORE_PATH,
    DOCS_DIR / "01_contexto_y_objetivos.md",
    DOCS_DIR / "02_fuentes_de_datos.md",
    DOCS_DIR / "03_pipeline_bronze_silver_gold.md",
    DOCS_DIR / "04_datasets_y_variables.md",
    DOCS_DIR / "05_modelos_entrenados.md",
    DOCS_DIR / "06_resultados_modelado.md",
    DOCS_DIR / "07_arquitectura_final.md",
    DOCS_DIR / "08_limitaciones_y_trabajo_futuro.md",
    DOCS_DIR / "09_reproducibilidad.md",
    DOCS_DIR / "notebooks_index.md",
    DOCS_DIR / "memoria_outline.md",
    DOCS_DIR / "github_checklist.md",
    DOCS_DIR / "bibliografia_base.md",
    DOCS_DIR / "documentation_manifest.csv",
]

missing = [str(p) for p in required_files if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Faltan archivos requeridos: " + json.dumps(missing, indent=2, ensure_ascii=False))

if README_PATH.stat().st_size < 500:
    raise ValueError("README.md parece demasiado pequeño.")

if len(list(DOCS_DIR.glob("*.md"))) < 10:
    raise ValueError("Hay pocos documentos Markdown en docs/.")

print("\nArchivos principales:")
for p in required_files:
    print("-", p)

print("\n✅ Pack de documentación generado correctamente.")

,path,size_kb,modified
0,.gitignore,0.44,2026-05-21T11:24:50
1,README.md,1.82,2026-05-21T11:24:50
2,docs/01_contexto_y_objetivos.md,0.72,2026-05-21T11:24:50
3,docs/02_fuentes_de_datos.md,0.37,2026-05-21T11:24:50
4,docs/03_pipeline_bronze_silver_gold.md,0.41,2026-05-21T11:24:50
5,docs/04_datasets_y_variables.md,0.42,2026-05-21T11:24:50
6,docs/05_modelos_entrenados.md,0.49,2026-05-21T11:24:50
7,docs/06_resultados_modelado.md,2.34,2026-05-21T11:24:50
8,docs/07_arquitectura_final.md,0.37,2026-05-21T11:24:50
9,docs/08_limitaciones_y_trabajo_futuro.md,0.51,2026-05-21T11:24:50



Archivos principales:
- /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/README.md
- /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/.gitignore
- /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/01_contexto_y_objetivos.md
- /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/02_fuentes_de_datos.md
- /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/03_pipeline_bronze_silver_gold.md
- /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/04_datasets_y_variables.md
- /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/05_modelos_entrenados.md
- /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/06_resultados_modelado.md
- /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs/07_arquitectura_final.md
- /Users/rauljimenez/Development/Projects/AI-Projects/deep-wave-canarias/docs

## Resultado esperado

Al final debe aparecer:

```text
✅ Pack de documentación generado correctamente.
```

Después puedes revisar `README.md`, `docs/` y hacer commit en GitHub.